# Stage 1 - Baseline system with Projection-B Alignment Training

Trains the MLP Projection-B to align CLIP ViT-L or DINOv2 ViT-L image features into Qwen 2.5 7B's word embedding space, using the LLaVA 1.5 558K image-caption dataset.

Alignment Training for Projection head B: Autoregressive next-token caption prediction through teacher forcing
Loss: Autoregressive cross-entropy over caption tokens only.  
Image token positions are masked with -100 so they contribute zero gradient.

Output: A trained projection_head_b.



In [ ]:
import os
import sys
from pathlib import Path

# Set these before importing any `reva` modules. Update the placeholder paths
# to match your machine or shared Jupyter environment.
os.environ["REVA_DATA_ROOT"] = "/path/to/your/data/root"
os.environ["REVA_HF_CACHE_ROOT"] = "/path/to/your/hf_cache/root"
os.environ["REVA_CHECKPOINT_ROOT"] = "/path/to/your/checkpoints/root"
os.environ["REVA_EVAL_RESULTS_ROOT"] = "/path/to/your/eval_results/root"
os.environ["REVA_REGION_DATA_ROOT"] = "/path/to/your/region_data/root"
os.environ["REVA_DECONTAMINATION_ROOT"] = "/path/to/your/decontamination/root"
os.environ["REVA_GROUNDING_DINO_ROOT"] = "/path/to/your/groundingdino/root"
os.environ["REVA_VQAV2_ROOT"] = "/path/to/your/vqav2/root"
os.environ["REVA_TEST_IMAGES_ROOT"] = "/path/to/your/test_images/root"

hf_cache_root = Path(os.environ["REVA_HF_CACHE_ROOT"]).expanduser()
os.environ["HF_HOME"] = str(hf_cache_root)
os.environ["HF_HUB_CACHE"] = str(hf_cache_root / "hub")
os.environ["HF_DATASETS_CACHE"] = str(hf_cache_root / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(hf_cache_root / "hub")

for var_name in (
    "REVA_DATA_ROOT",
    "REVA_HF_CACHE_ROOT",
    "REVA_CHECKPOINT_ROOT",
    "REVA_EVAL_RESULTS_ROOT",
    "REVA_REGION_DATA_ROOT",
    "REVA_DECONTAMINATION_ROOT",
    "REVA_GROUNDING_DINO_ROOT",
    "REVA_VQAV2_ROOT",
    "REVA_TEST_IMAGES_ROOT",
    "HF_HOME",
    "HF_HUB_CACHE",
    "HF_DATASETS_CACHE",
    "TRANSFORMERS_CACHE",
):
    print(f"{var_name} = {os.environ.get(var_name)}")


def find_reva_project_root(start: Path) -> Path:
    override = os.environ.get("REVA_PROJECT_DIR")
    if override:
        return Path(override).expanduser().resolve()

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "reva" / "evaluation.py").is_file() and (candidate / "reva" / "config.py").is_file():
            return candidate

    raise FileNotFoundError(
        "Could not locate the ReVA project root from the current working directory. "
        "Set REVA_PROJECT_DIR to your cloned repo path."
    )


PROJECT_ROOT = find_reva_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())


## Setup environment

In [ ]:
import os

os.makedirs(os.environ.get('REVA_HF_CACHE_ROOT') or os.path.expanduser('~/reva-data/hf_cache'), exist_ok=True) # for storing models, tokenizers and datasets locally from Huggingface
os.makedirs(os.environ.get('REVA_CHECKPOINT_ROOT') or os.path.expanduser('~/reva-data/checkpoints'), exist_ok=True) # for storing the checkpoints during Projection Head B's training 

# Configuring environment variables
os.environ['HF_HOME'] = os.environ.get('REVA_HF_CACHE_ROOT') or os.path.expanduser('~/reva-data/hf_cache')
os.environ['HF_DATASETS_CACHE'] = os.environ.get('REVA_HF_CACHE_ROOT') or os.path.expanduser('~/reva-data/hf_cache') + '/datasets'
os.environ['TRANSFORMERS_CACHE'] = os.environ.get('REVA_HF_CACHE_ROOT') or os.path.expanduser('~/reva-data/hf_cache') + '/hub'

print('Free in /tmp :', end=' ')
!df -h /tmp | awk 'NR==2{print $4}'

## Imports

In [ ]:
import torch
from pathlib import Path
from PIL import Image

from reva.config import TrainingConfig, EvalConfig
from reva.models import load_frozen_vit, load_frozen_qwen, load_frozen_dinov2, build_projection_head, ProjectionHeadB
from reva.dataset import download_and_prepare_llava, build_dataloader
from reva.training import (EarlyStopper, LossTracker, build_optimiser_and_scheduler, save_projection_b_checkpoint, 
                        load_projection_b_from_checkpoint, train_projection_b)
from reva.inference import generate_caption_for_image, interactive_inference_loop
from reva.evaluation import (download_gqa_val, download_gqa_testdev, run_gqa_evaluation, download_vqav2_val, 
                        download_vqav2_testdev, run_vqav2_val_inference, run_vqav2_testdev_inference)

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

## Initialize Training Configuration for Projection Head B

In [ ]:
config = TrainingConfig()
config.checkpoint_output_dir.mkdir(parents=True, exist_ok=True)
print(config)

## Load frozen models

In [ ]:
frozen_vit, clip_image_processor = load_frozen_vit(config)

In [ ]:
frozen_qwen, qwen_tokenizer = load_frozen_qwen(config)

In [ ]:
frozen_dino, dino_image_processor = load_frozen_dinov2(config)
frozen_vit, clip_image_processor = frozen_dino, dino_image_processor

## Build trainable projection head

In [ ]:
projection_head_b = build_projection_head(config)

## Download dataset

In [ ]:
raw_dataset, images_root_dir = download_and_prepare_llava(config)

## Data decontamination (removes COCO & VG overlaps)

In [ ]:
# Build reference image paths (COCO + VG) before calling decontaminate_dataset
# Uncomment and set these paths if you have COCO/VG images downloaded:
# coco_image_paths = list((vqav2_dir / 'train2014').glob('*.jpg')) + ...
# vg_image_paths = list(gqa_images_dir.glob('*.jpg'))
# all_ref_paths = coco_image_paths + vg_image_paths

# raw_dataset, phash_removed, sscd_removed, corrupt = decontaminate_dataset(
#     raw_dataset, images_root_dir, all_ref_paths, config
# )
print("Decontamination skipped, using full raw_dataset")

## Build DataLoader

In [ ]:
train_dataloader = build_dataloader(raw_dataset, clip_image_processor, qwen_tokenizer, images_root_dir, config)

## Initialise training utilities

In [ ]:
early_stopper = EarlyStopper(
    patience=config.early_stopping_patience,
    min_delta=config.early_stopping_min_delta,
    ema_factor=config.loss_ema_smoothing_factor,
)

loss_tracker = LossTracker(
    log_json_path=config.log_json_path,
    plot_output_path=config.plot_output_path,
)

optimiser, lr_scheduler = build_optimiser_and_scheduler(projection_head_b, train_dataloader, config)

## Train

In [ ]:
import os
os.makedirs(os.environ.get('REVA_CHECKPOINT_ROOT') or os.path.expanduser('~/reva-data/checkpoints'), exist_ok=True)
print("Checkpoint directory ready")

In [ ]:
print('Starting Projection-B alignment training...')

final_global_step, stop_reason = train_projection_b(
    frozen_vit=frozen_vit,
    projection_head_b=projection_head_b,
    frozen_qwen=frozen_qwen,
    train_dataloader=train_dataloader,
    optimiser=optimiser,
    lr_scheduler=lr_scheduler,
    early_stopper=early_stopper,
    loss_tracker=loss_tracker,
    config=config
)

print(f'\nTraining complete.')
print(f'Stop reason: {stop_reason}')
print(f'Total steps: {final_global_step:,}')

## Save final checkpoint

In [ ]:
import torch

final_loss = loss_tracker.smoothed_losses[-1] if loss_tracker.smoothed_losses else float('nan')

final_checkpoint_path = save_projection_b_checkpoint(
    projection_head_b=projection_head_b,
    optimiser=optimiser,
    lr_scheduler=lr_scheduler,
    global_step=final_global_step,
    running_avg_loss=final_loss,
    config=config
)
weights_only_path = config.checkpoint_output_dir / 'projection_b_final_weights.pt'
torch.save(projection_head_b.state_dict(), weights_only_path)
print(f'Weights-only file: {weights_only_path}')

loss_tracker.plot_and_save()
loss_tracker.save_json_log()

from IPython.display import Image as IPImage, display
display(IPImage(filename=str(config.plot_output_path)))

## Test: Generate captions for a few images

In [ ]:
print('Generating captions for 3 validation images...\n')
projection_head_b.eval()

for check_idx in range(3):
    sample = raw_dataset[check_idx]
    ground_truth = sample['conversations'][1]['value']

    image_path = Path(images_root_dir) / sample['image']
    pil_image = Image.open(image_path).convert('RGB')

    generated = generate_caption_for_image(
        pil_image=pil_image,
        frozen_vit=frozen_vit,
        projection_head_b=projection_head_b,
        frozen_qwen=frozen_qwen,
        clip_image_processor=clip_image_processor,
        qwen_tokenizer=qwen_tokenizer,
        config=config,
    )
    print(f'Sample {check_idx + 1}')
    print(f'Ground truth: {ground_truth}')
    print(f'Generated: {generated}')
    print()

## Load trained projection head (Projection-B)

In [ ]:
eval_config = EvalConfig()
eval_config.results_dir.mkdir(parents=True, exist_ok=True)

# Load best projection head
projection_head_b = ProjectionHeadB(
    input_dim=config.vit_patch_feature_dim,
    output_dim=config.qwen_embedding_dim,
).to(config.device, dtype=config.compute_dtype)
projection_head_b.load_state_dict(
    torch.load(
        #'~/reva-data/checkpoints/projection_b_best_weights.pt', 
        #'/home/jovyan/teaching_material/MScProject/projectionB-best-weights-CLIPViT-L14-224/projection_b_best_weights.pt',
        #'/home/jovyan/teaching_material/MScProject/projectionB-best-weights-CLIPViT-L14-336/projection_b_best_weights.pt',
        #'/home/jovyan/teaching_material/MScProject/projectionB-new-best-weights-CLIPViT-L14-336.pt',
        os.environ.get('REVA_PROJECTION_B_WEIGHTS', 'projection_b_best_weights.pt'),
        #'/home/jovyan/teaching_material/MScProject/projectionB-best-weights-DINOv2ViT-L14/projection_b_best_weights.pt',
        map_location=config.device
    )
)
projection_head_b.eval()

## Zero-shot evaluation (GQA val-balanced)

In [ ]:
# import urllib.request, zipfile, io

# req = urllib.request.Request('https://nlp.stanford.edu/data/gqa/questions1.2.zip', headers={'User-Agent': 'Mozilla/5.0'})
# r = urllib.request.urlopen(req)
# z = zipfile.ZipFile(io.BytesIO(r.read()))

# print('\n'.join(f'{f.filename}: {f.file_size} bytes' for f in z.infolist() if '_questions.json' in f.filename))


In [ ]:
# Temp
gqa_dir = Path(os.environ.get('REVA_GQA_ROOT') or os.path.expanduser('~/reva-data/gqa'))
gqa_samples, gqa_images_dir = download_gqa_val(gqa_dir)

gqa_results = run_gqa_evaluation(
    gqa_samples=gqa_samples,
    gqa_images_dir=gqa_images_dir,
    frozen_vit=frozen_vit,
    projection_head_b=projection_head_b,
    frozen_qwen=frozen_qwen,
    clip_image_processor=clip_image_processor,
    qwen_tokenizer=qwen_tokenizer,
    eval_config=eval_config,
    desc="GQA val-balanced"
)

## Zero-shot inference (GQA testdev-balanced)

In [ ]:
# New
gqa_dir = Path(os.environ.get('REVA_GQA_ROOT') or os.path.expanduser('~/reva-data/gqa'))
gqa_samples, gqa_images_dir = download_gqa_testdev(gqa_dir)

gqa_results = run_gqa_evaluation(
    gqa_samples=gqa_samples,
    gqa_images_dir=gqa_images_dir,
    frozen_vit=frozen_vit,
    projection_head_b=projection_head_b,
    frozen_qwen=frozen_qwen,
    clip_image_processor=clip_image_processor,
    qwen_tokenizer=qwen_tokenizer,
    eval_config=eval_config,
    desc="GQA testdev-balanced"
)

## Zero-shot evaluation (VQAv2 val)

In [ ]:
vqav2_dir = Path(os.environ.get('REVA_VQAV2_ROOT') or os.path.expanduser('~/reva-data/vqav2'))
vqav2_val_samples, val_images_dir = download_vqav2_val(vqav2_dir)

val_results = run_vqav2_val_inference(
    vqav2_samples=vqav2_val_samples,
    val_images_dir=val_images_dir,
    frozen_vit=frozen_vit,
    projection_head_b=projection_head_b,
    frozen_qwen=frozen_qwen,
    clip_image_processor=clip_image_processor,
    qwen_tokenizer=qwen_tokenizer,
    eval_config=eval_config,
    results_path=Path(os.environ.get('REVA_EVAL_RESULTS_ROOT') or os.path.expanduser('~/reva-data/eval_results')) / 'vqav2_val_results.json',
)

In [ ]:
import json as json
# Inspect 10 samples to find where scoring is failing
with open(Path(os.environ.get("REVA_EVAL_RESULTS_ROOT") or os.path.expanduser("~/reva-data/eval_results")) / "vqav2_val_results.json") as f:
    saved = json.load(f)

for pred in saved['predictions'][:10]:
    print(f"Question: {pred['question']}")
    print(f"Predicted answer : '{pred['predicted_answer']}'")
    print(f"GT answers: {pred['ground_truth']}")
    print(f"Soft score: {pred['soft_score']}")
    print()

# Check score distribution
scores = [p['soft_score'] for p in saved['predictions']]
print(f"Score distribution:")
print(f"  Score = 0.000: {sum(1 for s in scores if s == 0.0):,}  ({sum(1 for s in scores if s == 0.0)/len(scores)*100:.1f}%)")
print(f"  Score = 0.333: {sum(1 for s in scores if abs(s-0.333)<0.01):,}")
print(f"  Score = 0.667: {sum(1 for s in scores if abs(s-0.667)<0.01):,}")
print(f"  Score = 1.000: {sum(1 for s in scores if s == 1.0):,}  ({sum(1 for s in scores if s == 1.0)/len(scores)*100:.1f}%)")

### Testing the system using VQA scoring (Null Image tests & OOD tests)

In [ ]:
import json
import numpy as np
from PIL import Image
from pathlib import Path

# Null-image testing
N_SAMPLES = 200  

with open(Path(os.environ.get("REVA_EVAL_RESULTS_ROOT") or os.path.expanduser("~/reva-data/eval_results")) / "vqav2_val_results.json") as f:
    saved = json.load(f)

real_preds = saved['predictions'][:N_SAMPLES]
null_image = Image.fromarray(np.zeros((336, 336, 3), dtype=np.uint8))  # black

null_scores = []
for pred in real_preds:
    generated = generate_caption_for_image(
        pil_image=null_image,
        frozen_vit=frozen_vit,
        projection_head_b=projection_head_b,
        frozen_qwen=frozen_qwen,
        clip_image_processor=clip_image_processor,
        qwen_tokenizer=qwen_tokenizer,
        config=config,
    )
    gt_answers = pred['ground_truth']
    matches = gt_answers.count(generated.strip().lower())
    soft_score = min(1.0, matches / 3.0)
    null_scores.append(soft_score)

real_avg = sum(p['soft_score'] for p in real_preds) / N_SAMPLES
null_avg = sum(null_scores) / N_SAMPLES

print(f"Real image accuracy : {real_avg:.3f}")
print(f"Null image accuracy : {null_avg:.3f}")
print()
print(f"Question: {pred['question']}")
print(f"Null output: '{generated}'")
print(f"Ground truth: {pred['ground_truth']}")
print()

In [ ]:
from datasets import load_dataset

distractor_ds = load_dataset("zh-plus/tiny-imagenet", split="valid", streaming=True)
distractors = [sample['image'].convert('RGB') for sample, _ in zip(distractor_ds, range(20))]

null_scores = []
for i, pred in enumerate(real_preds[:10]):  # just 10 first to diagnose
    wrong_image = distractors[i % len(distractors)]
    generated = generate_caption_for_image(
        pil_image=wrong_image,
        frozen_vit=frozen_vit,
        projection_head_b=projection_head_b,
        frozen_qwen=frozen_qwen,
        clip_image_processor=clip_image_processor,
        qwen_tokenizer=qwen_tokenizer,
        config=config,
    )
    print(f"Q: {pred['question']}")
    print(f"Wrong image output: '{generated}'")
    print(f"GT: {pred['ground_truth'][:3]}")
    print()

## Zero-shot inference (VQAv2 testdev)

In [ ]:
vqav2_dir = Path(os.environ.get('REVA_VQAV2_ROOT') or os.path.expanduser('~/reva-data/vqav2'))
vqav2_testdev_samples, test_images_dir = download_vqav2_testdev(vqav2_dir)

testdev_predictions = run_vqav2_testdev_inference(
    vqav2_samples=vqav2_testdev_samples,
    test_images_dir=test_images_dir,
    frozen_vit=frozen_vit,
    projection_head_b=projection_head_b,
    frozen_qwen=frozen_qwen,
    clip_image_processor=clip_image_processor,
    qwen_tokenizer=qwen_tokenizer,
    eval_config=eval_config,
    submission_path=Path(os.environ.get('REVA_EVAL_RESULTS_ROOT') or os.path.expanduser('~/reva-data/eval_results')) / 'vqav2_testdev_submission.json',
)

## Interactive inference for system (zero-shot)

### Using projectionB-best-weights-CLIPViT-L14-336

In [ ]:
interactive_inference_loop(
    frozen_vit=frozen_vit,
    projection_head_b=projection_head_b,
    frozen_qwen=frozen_qwen,
    clip_image_processor=clip_image_processor,
    qwen_tokenizer=qwen_tokenizer,
    config=config
)